# CardioSentry — synthetic ECG image generation (Colab)

Generates the full synthetic paper-ECG corpus from the **v3** harmonized signals.

| | |
|---|---|
| input | `datasets/harmonized.tar` (v3 signals) + `datasets/cardiosentry_pipeline.tar.gz` (patched code) |
| output | `experiments/synthetic_images_v1/` (corpus archives) and `runs/synthetic_images_v1/` (index, logs) |
| expected | ~37,225 records -> ~129,376 images, 8-12 h on 12 vCPU |

**The GPU is not used.** Nothing in this pipeline touches CUDA; the L4 runtime is
only a way to get 12 vCPUs. Section 0 disables CUDA so TensorFlow (imported by the
kit but never called) does not waste time initialising it.

**Everything is resumable.** Stages 2-4 skip work already on disk, so a Colab
disconnect costs only the in-flight chunk. Re-run the same cells.

---
## How to use
1. Run **section 1 (preflight)** first. It hard-stops on any problem *before* the
   8-hour job starts. Do not skip it.
2. Run section 2 (smoke test, ~2 min) and eyeball the image.
3. Run section 3 (full build).
4. Run section 4 (package to Drive).


## 0. Runtime setup


In [ ]:
import os, sys, subprocess, time, json, importlib, importlib.util
from pathlib import Path

# Nothing here uses the GPU. Hiding it stops TF from initialising CUDA on every
# one of the ~100 kit invocations, which is pure overhead.
os.environ['CUDA_VISIBLE_DEVICES'] = ''

DRIVE   = Path('/content/drive/MyDrive/CardioSentry')
DATASETS= DRIVE/'datasets'
CODES   = DRIVE/'codes'/'synthetic_image_generation'
EXPDIR  = DRIVE/'experiments'/'synthetic_images_v1'
RUNDIR  = DRIVE/'runs'/'synthetic_images_v1'

WORK    = Path('/content/work')          # LOCAL disk - never build on Drive
CODE    = WORK/'code'
DATA    = WORK/'data'/'releases'/'v3'
BUILD   = WORK/'build'

from google.colab import drive
drive.mount('/content/drive')
print('mounted')


In [ ]:
# Build parameters. Change these BEFORE the preflight, not after.
PARAMS = dict(
    ECG_RELEASE     = 'v3',
    ECG_WORKERS     = str(os.cpu_count() or 12),
    ECG_IMAGE_BUILD = str(BUILD),
    ECGKIT_DIR      = str(CODE/'ecg-image-kit'),
    ECGKIT_PY       = sys.executable,
    ECG_ANNOTATIONS = '1',   # '0' saves ~45 GB and ~10% time; drop only if you will
                             # never do segmentation / digitisation work
    # Deliberately NOT set, so the config defaults apply:
    #   ECG_LOWPASS_HZ  -> 0.05-40 Hz (paired with 150 DPI; see config.py)
    #   ECG_PRINT_HEADER-> off (no ID/Name/Date/Age/Sex)
    #   ECG_ROW_GAP     -> 4.4   ECG_RHYTHM_GAP -> 5.5
    #   ECG_DPI         -> 150
)
os.environ.update(PARAMS)
for k, v in PARAMS.items(): print(f'{k:18s} {v}')


## 1. Preflight

Every check below hard-stops the notebook. The point is to fail in the first two
minutes rather than eight hours in.


In [ ]:
FAIL = []
def check(name, ok, detail=''):
    print(('  OK   ' if ok else '  FAIL ') + name + (f'  -- {detail}' if detail else ''))
    if not ok: FAIL.append(name)

def die():
    """Stop the notebook if anything in this section failed."""
    if FAIL:
        msg = 'PREFLIGHT FAILED:\n  - ' + '\n  - '.join(FAIL)
        FAIL.clear()          # so a re-run after a fix starts clean
        raise SystemExit(msg)
    print('  -> section OK')

def section(title):
    FAIL.clear()              # each section judges only its own checks
    print(title)

section('[1] Drive layout')
for p in (DRIVE, DATASETS):
    check(f'{p} exists', p.is_dir())
for p in (CODES, EXPDIR, RUNDIR):
    p.mkdir(parents=True, exist_ok=True)
    check(f'{p.relative_to(DRIVE)} writable', os.access(p, os.W_OK))
die()


In [ ]:
section('[2] Input archives present in datasets/')
SIG_TAR  = DATASETS/'harmonized.tar'
CODE_TAR = DATASETS/'cardiosentry_pipeline.tar.gz'
check('harmonized.tar', SIG_TAR.is_file(), f'{SIG_TAR.stat().st_size/2**30:.2f} GB' if SIG_TAR.is_file() else 'missing')
check('cardiosentry_pipeline.tar.gz', CODE_TAR.is_file(),
      f'{CODE_TAR.stat().st_size/2**20:.0f} MB' if CODE_TAR.is_file() else 'upload it to datasets/')
die()


In [ ]:
section('[3] Local disk and RAM')
st = os.statvfs('/content')
free_gb = st.f_bavail*st.f_frsize/2**30

# Per-unit sizes MEASURED on a real 5-record build at the current settings
# (150 DPI, 4 columns, 0.05-40 Hz):
#
#   rendered/  PNG   255 KB    JSON 1,818 KB   <- the JSON is 88% of rendered/
#   images/    JPEG  590 KB    JSON   365 KB
#
# An earlier version of this check counted rendered/ as PNG-only. That
# understated the peak by 65 GB and a run died at stage 4 with ENOSPC after
# eight hours of rendering. The kit's annotation sidecar is SEVEN TIMES the
# size of the image it describes - do not collapse these back into one constant.
N_REC, N_IMG = 37_225, 129_376
ANN = os.environ['ECG_ANNOTATIONS'] == '1'
GB  = lambda kb, n: kb*n/2**20

sig       = 4.5                                  # extracted harmonized signals
staged    = GB(132,  N_REC)
rend_png  = GB(255,  N_REC)
rend_json = GB(1818, N_REC) if ANN else 0.0
img_jpg   = GB(590,  N_IMG)
img_json  = GB(365,  N_IMG) if ANN else 0.0

# The peak is inside stage 4: rendered/ is still whole while images/ reaches
# full size. By then the cleanup cell has dropped the signals and staged/.
peak = rend_png + rend_json + img_jpg + img_json
need = max(sig + staged + rend_png + rend_json, peak)
print(f'  stage 3 ends at {sig+staged+rend_png+rend_json:5.1f} GB'
      f'   stage 4 peaks at {peak:5.1f} GB')
check(f'free disk {free_gb:.0f} GB >= {need:.0f} GB peak', free_gb >= need,
      f'ECG_ANNOTATIONS=0 drops the peak to {rend_png+img_jpg:.0f} GB; or keep '
      f'them\n       through stage 3 and set DROP_RENDER_ANNOTATIONS before stage 4'
      if ANN else 'no room even without annotations')

ram = os.sysconf('SC_PAGE_SIZE')*os.sysconf('SC_PHYS_PAGES')/2**30
check(f'RAM {ram:.0f} GB', ram >= 8)
check(f'vCPUs {os.cpu_count()}', (os.cpu_count() or 0) >= 4)
die()


In [ ]:
section('[4] Extract code + signals to LOCAL disk')
# Reading 74,662 signal files through the Drive FUSE layer would take hours.
# Extract once to /content and never touch Drive again until packaging.
def untar(archive, dest, flag):
    dest.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['tar', flag, str(archive), '-C', str(dest)],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1500:]); raise SystemExit(f'failed to extract {archive.name}')

CODE = WORK/'code'
if not (CODE/'image_pipeline'/'config.py').exists():
    untar(CODE_TAR, CODE, '-xzf')
if not (DATA/'harmonized'/'harmonized_ecg_metadata.csv').exists():
    untar(SIG_TAR, DATA, '-xf')

PIPE = CODE/'image_pipeline'
CSV  = DATA/'harmonized'/'harmonized_ecg_metadata.csv'

# Optional CSV override. The metadata CSV changes far more often than the 4.4 GB
# of signals it describes - freezing the train/val/test split rewrites it, for
# one. Dropping the newer CSV into datasets/ avoids re-uploading the whole tar.
CSV_OVERRIDE = DATASETS/'harmonized_ecg_metadata.csv'
if CSV_OVERRIDE.is_file():
    import shutil
    shutil.copy2(CSV_OVERRIDE, CSV)
    print(f'  using CSV override from datasets/ ({CSV_OVERRIDE.stat().st_size/2**20:.0f} MB)')

check('image_pipeline/config.py', (PIPE/'config.py').is_file())
check('harmonized_ecg_metadata.csv', CSV.is_file())
sigdir = DATA/'harmonized'/'signals'
n_sig = len(list(sigdir.rglob('*.hea'))) if sigdir.is_dir() else 0
check(f'signal headers found: {n_sig:,}', n_sig > 30000, 'expected ~37,329')

# The frozen split. stage 1 REFUSES to invent one, by design: the split is a
# property of the dataset, so a build must never silently reassign it. If this
# fails, run  python scripts/freeze_splits.py --release v3  on the source
# release and put the resulting CSV in datasets/ as harmonized_ecg_metadata.csv.
import pandas as _pd
_h = _pd.read_csv(CSV, nrows=5)
has_split = 'split' in _h.columns
check('CSV has a split column', has_split,
      'run scripts/freeze_splits.py --release v3, then drop the CSV into datasets/')
if has_split:
    _s = _pd.read_csv(CSV, usecols=['split'])
    vc = _s.split.value_counts(dropna=False)
    print('   ', dict(vc))
    check('split is fully populated', _s.split.notna().all() and (_s.split != '').all())
    check('all three folds present', set(vc.index) >= {'train','val','test'})
die()


In [ ]:
section('[5] Python dependencies')
import importlib, importlib.util

# This list was derived EMPIRICALLY, by blocking each candidate and re-running
# a real render until it broke. It is not the kit's requirements.txt, which
# pins tensorflow, spacy, seaborn, keras, imutils, validators, bs4 and
# scikit-learn - all of which patch_kit.py's lazy-deps patch removes from the
# import path, because the helpers that need them are gated behind flags this
# pipeline never sets.
#
# imgaug itself hard-requires cv2 and skimage; both are Colab preinstalls.
MODS = ('numpy','scipy','pandas','wfdb','PIL','matplotlib',
        'imgaug','cv2','skimage','qrcode','imageio','yaml')
PIP  = {'PIL':'pillow','cv2':'opencv-python','skimage':'scikit-image',
        'yaml':'pyyaml','imgaug':'imgaug','wfdb':'wfdb','qrcode':'qrcode',
        'imageio':'imageio'}

missing = [m for m in MODS if not importlib.util.find_spec(m)]
if missing:
    pkgs = sorted({PIP.get(m, m) for m in missing})
    print('  installing:', pkgs)
    subprocess.run([sys.executable,'-m','pip','-q','install',*pkgs], check=False)
    importlib.invalidate_caches()

# numpy 2 removed np.sctypes; imgaug 0.4.0 reads it at module level. The kit's
# entry point carries the same shim (patch_kit.py) - this copy only lets the
# notebook process import imgaug to verify it is installed.
import numpy as _np
if not hasattr(_np, 'sctypes'):
    _np.sctypes = {
        'int':     [_np.int8, _np.int16, _np.int32, _np.int64],
        'uint':    [_np.uint8, _np.uint16, _np.uint32, _np.uint64],
        'float':   [_np.float16, _np.float32, _np.float64],
        'complex': [_np.complex64, _np.complex128],
        'others':  [bool, object, bytes, str, _np.void],
    }
    print(f'  applied np.sctypes shim (numpy {_np.__version__})')
print(f'  numpy {_np.__version__}')

# IMPORT each one. find_spec only proves a module EXISTS and never executes it,
# which is exactly how imgaug/numpy-2 and the missing `validators` both passed
# preflight and then killed every render two minutes later.
for m in MODS:
    try:
        importlib.import_module(m)
        check(m, True)
    except Exception as e:
        check(m, False, f'{type(e).__name__}: {e}')
die()


In [ ]:
section('[6] Kit patches — the build is WRONG without these')
sys.path.insert(0, str(PIPE))
os.chdir(PIPE)
for m in list(sys.modules):
    if m in ('config','patch_kit'): del sys.modules[m]
import config as C, patch_kit
print(subprocess.run([sys.executable,'patch_kit.py'],capture_output=True,text=True).stdout)
importlib.reload(patch_kit)
check('all kit patches applied', patch_kit.is_patched(),
      'patch_kit reported FAILED above -- do not proceed')
die()


In [ ]:
section('[7] Effective settings — read these carefully')
importlib.reload(C)
print(f'  band-pass       {C.BANDPASS[0]}-{C.BANDPASS[1]} Hz')
print(f'  printed header  {C.PRINT_HEADER}')
print(f'  annotations     {C.ANNOTATIONS}')
print(f'  DPI             {C.DPI}')
print(f'  row / rhythm gap{C.ROW_GAP} / {C.RHYTHM_GAP}  (x10 mm)')
print(f'  workers         {C.WORKERS}')
print(f'  variants        {C.VARIANTS_PER_SPLIT}')
check('band-pass is 0.05-40 Hz', C.BANDPASS == (0.05, 40.0),
      'the 40 Hz band is PAIRED with 150 DPI: image Nyquist is 73.8 Hz, and a\n       wider band renders mains hum concentrated in CUHK, the only source of\n       STEMI co-morbidity')
check('printed header is OFF', C.PRINT_HEADER is False,
      'ON would drop ~17,875 age-less records and zero the STEMI+AF+LVH class')
_f = C.KIT_RENDER_FLAGS
_ncol = _f[_f.index('--num_columns')+1] if '--num_columns' in _f else None
check(f'4 leads per row (got {_ncol})', _ncol == '4',
      'more columns shortens each panel: 10 s is split across them')
die()

# Probe the KIT's own import chain in a subprocess, exactly as stage 3 runs it.
# This is what proves the numpy-2 shim is in place before anything long starts.
probe = subprocess.run([sys.executable, '-c',
    'import gen_ecg_images_from_data_batch as m; print("kit entry point OK")'],
    cwd=C.KIT, capture_output=True, text=True,
    env={**os.environ, 'TF_CPP_MIN_LOG_LEVEL': '3'})
check('kit entry point imports (numpy-2 shim works)', probe.returncode == 0,
      probe.stderr.strip().splitlines()[-1] if probe.returncode else '')
die()


## 2. Smoke test (~2 min)

Renders 5 records that have **no recorded age**, including the triple-positive
class. This is the specific case that two separate bugs used to drop silently, so
it is the one worth proving before an 8-hour run.


In [ ]:
import pandas as pd
SMOKE = WORK/'smoke'; SMOKE.mkdir(parents=True, exist_ok=True)
df = pd.read_csv(CSV, low_memory=False)
here = CSV.parent
noage = df[df.age.isna() & (df.validation_status=='VALID')]
picks = [noage[noage.cardiosentry_label==l].iloc[0]
         for l in ('STEMI+AF+LVH','STEMI+LVH','STEMI+AF','AF+LVH','LVH')
         if len(noage[noage.cardiosentry_label==l])]
sub = pd.DataFrame(picks).copy()
for c in ('signal_path','header_path','data_path'):
    sub[c] = sub[c].map(lambda p: str((here/p).resolve()))
sub['split'] = 'test'
sub.to_csv(SMOKE/'smoke.csv', index=False)
print(sub[['global_record_id','source_dataset','cardiosentry_label','age','sex']].to_string(index=False))


In [ ]:
LOGS = WORK/'logs'; LOGS.mkdir(parents=True, exist_ok=True)

def run(script, *args, env=None, quiet=False, tag=None):
    """Run a pipeline stage, STREAMING its output live and teeing it to a log.

    Not subprocess.run(capture_output=True): that buffers everything until the
    process exits, so a 5-8 hour stage 3 would show nothing at all until it was
    over. Popen + line iteration prints progress as it happens, and -u stops
    Python block-buffering its stdout when writing to a pipe instead of a tty.

    Every line also lands in WORK/logs/<stage>.log, so the Colab terminal can
        tail -f /content/work/logs/stage3.log
    and watch the same output from outside the notebook.
    """
    name = tag or script.replace('.py','')
    log  = LOGS/f'{name}.log'
    cmd  = [sys.executable, '-u', script, *args]
    started = time.time()
    lines = []
    with open(log, 'w') as fh:
        proc = subprocess.Popen(cmd, cwd=PIPE, env={**os.environ, **(env or {})},
                                stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            fh.write(line); fh.flush()
            lines.append(line)
            if not quiet:
                print(line, end='', flush=True)
        proc.wait()
    out = ''.join(lines).strip()
    mins = (time.time()-started)/60
    print(f'  [{name}] finished in {mins:.1f} min  ->  {log}')
    if proc.returncode:
        print(out[-2000:]); raise SystemExit(f'{script} failed (see {log})')
    return out

SMOKE_ENV = {'ECG_IMAGE_BUILD': str(SMOKE/'build')}
W = os.environ['ECG_WORKERS']
run('stage1_manifest_from_csv.py', '--input-csv', str(SMOKE/'smoke.csv'),
    env=SMOKE_ENV, tag='smoke_stage1')
run('stage2_transcode.py', env=SMOKE_ENV, tag='smoke_stage2')
run('stage3_render.py', '-j', W, env=SMOKE_ENV, tag='smoke_stage3')
run('stage4_augment.py', '-j', W, env=SMOKE_ENV, tag='smoke_stage4')


In [ ]:
section('[smoke] verify')
imgs = sorted((SMOKE/'build'/'images').rglob('*.jpg'))
check(f'5 images produced (got {len(imgs)})', len(imgs) == 5,
      'fewer means records are still being dropped')

import numpy as np
from PIL import Image as PILImage
ppmm = C.DPI/25.4
pngs = sorted((SMOKE/'build'/'rendered').rglob('*.png'))
check('clean renders exist', len(pngs) > 0)
if pngs:
    g = np.asarray(PILImage.open(pngs[0]).convert('L'), float)
    band = g[int(14*ppmm):int(30*ppmm), int(15*ppmm):int(95*ppmm)]
    check('no header text on the sheet', band.min() >= 95,
          f'darkest pixel {band.min():.0f}; text would be <70')
die()

im = PILImage.open(imgs[0])
display(im.resize((1100, int(1100*im.height/im.width))))


## 3. Full build (8-12 h)

Safe to re-run after a disconnect — every stage skips finished work.

Stage 1 prints the record count and split. **Sanity-check it before letting
stage 3 start:** you want ~37,225 records and a `STEMI+AF+LVH` row that is not zero.


In [ ]:
# Full build. run() now reads os.environ at call time, so ECG_IMAGE_BUILD is
# BUILD (set in section 0), not the smoke directory.
assert os.environ['ECG_IMAGE_BUILD'] == str(BUILD), os.environ['ECG_IMAGE_BUILD']
t0 = time.time()
out1 = run('stage1_manifest_from_csv.py', '--input-csv', str(CSV))


In [ ]:
section('[gate] manifest sanity')
# Refuse to burn 8 hours on a manifest that lost the multi-label classes.
man = pd.read_csv(BUILD/'image_manifest.csv')
counts = man.cls.value_counts()
print(counts.to_string()); print(f'\ntotal {len(man):,}')
check(f'record count {len(man):,} >= 35,000', len(man) >= 35000)
check(f"STEMI+AF+LVH present ({counts.get('STEMI+AF+LVH',0)})", counts.get('STEMI+AF+LVH',0) > 0)
check(f"AF+LVH >= 2,000 ({counts.get('AF+LVH',0)})", counts.get('AF+LVH',0) >= 2000)
die()


In [ ]:
section('[gate] stage 2')
out2 = run('stage2_transcode.py')
# Assert against config, not a hardcoded string: hardcoding the band is what
# left a stale '0.05-150 Hz' check here after the default was changed.
_band = f'{C.BANDPASS[0]:g}-{C.BANDPASS[1]:g} Hz'
check(f'stage 2 used the configured band ({_band})', _band in out2,
      out2.splitlines()[0] if out2 else 'no output')
check('no records lost to missing_metadata', 'missing_metadata' not in out2)
die()


In [ ]:
# The long one, ~5-8 h. Safe to re-run after a disconnect.
out3 = run('stage3_render.py', '-j', os.environ['ECG_WORKERS'])
print(f'elapsed so far: {(time.time()-t0)/3600:.1f} h')


### Checkpoint after stage 3 — read this

Everything so far lives on `/content`, which is **ephemeral**. Reconnecting to the
same runtime keeps it; a *terminated* runtime wipes it. Stage 3 is the 5-8 hour
part of the job, so it is worth 9 GB and ~15 min of upload to make it survivable.

If the runtime is lost later, run the restore cell instead of re-rendering.


In [ ]:
CHECKPOINT = True     # set False to skip (you are then betting 5-8 h on the VM)

CKPT = EXPDIR/'checkpoint_rendered.tar'
if CHECKPOINT and not CKPT.exists():
    EXPDIR.mkdir(parents=True, exist_ok=True)
    sz = sum(f.stat().st_size for f in (BUILD/'rendered').rglob('*') if f.is_file())
    print(f'archiving {sz/2**30:.1f} GB of clean renders to Drive ...', flush=True)
    r = subprocess.run(['tar','-cf',str(CKPT),'-C',str(BUILD),
                        'rendered','image_manifest.csv','staged_map.csv'],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1500:]); raise SystemExit('checkpoint failed')
    print(f'  {CKPT.name}  {CKPT.stat().st_size/2**30:.1f} GB')
else:
    print('checkpoint skipped or already present')


In [ ]:
# RESTORE — only if the runtime was lost after stage 3. Re-run sections 0-1
# first (they re-extract the code and signals), then this, then stage 4.
RESTORE = False
if RESTORE:
    assert CKPT.exists(), f'no checkpoint at {CKPT}'
    BUILD.mkdir(parents=True, exist_ok=True)
    r = subprocess.run(['tar','-xf',str(CKPT),'-C',str(BUILD)],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1500:]); raise SystemExit('restore failed')
    n = len(list((BUILD/'rendered').rglob('*.png')))
    print(f'restored {n:,} clean renders - stage 4 will pick up from here')


### Reclaim disk before stage 4


In [ ]:
# RECLAIM DISK - run this after the stage-3 checkpoint, before stage 4.
#
# Stage 3 is done and archived to Drive, so everything it needed is dead weight:
#   data/releases/v3/harmonized   read only by stage 2 (done; stage 2 skips on a
#                                 matching manifest+band fingerprint)
#   build/staged                  read only by stage 3 (done)
# Both come back from datasets/harmonized.tar in ~15 min if ever needed again.
import shutil

def free_gb():
    s = os.statvfs('/content')
    return s.f_bavail*s.f_frsize/2**30

print(f'free before  {free_gb():.1f} GB')
for p in (DATA/'harmonized', BUILD/'staged'):
    if p.exists():
        shutil.rmtree(p); print(f'  removed {p}')
print(f'free after   {free_gb():.1f} GB')

# rendered/*.json is the kit's annotation output: 1.8 MB per record against
# 255 KB for the PNG beside it, ~65 GB in total. Stage 4 reads it to carry the
# lead bounding boxes onto the augmented image, and tolerates it being absent
# (it just writes has_annotation=0 into index.csv - see _recover_meta).
#
# Keeping it makes stage 4 peak at ~191 GB, which does not fit in the ~188 GB
# Colab gives you. Set this True to trade the corpus sidecars for the headroom;
# the originals are already inside checkpoint_rendered.tar on Drive, so stage 4
# can be re-run later against a restored checkpoint to get them back.
DROP_RENDER_ANNOTATIONS = False

if DROP_RENDER_ANNOTATIONS:
    assert CKPT.exists(), 'refusing to delete annotations that are not checkpointed'
    n = 0
    for j in (BUILD/'rendered').rglob('*.json'):
        j.unlink(); n += 1
    print(f'  deleted {n:,} render annotations -> free {free_gb():.1f} GB')


In [ ]:
# ~2-4 h. Safe to re-run.
out4 = run('stage4_augment.py', '-j', os.environ['ECG_WORKERS'], quiet=True)
print(f'TOTAL elapsed: {(time.time()-t0)/3600:.1f} h')


In [ ]:
out5 = run('stage5_verify.py')


### Checking CPU utilisation

Run this in a **separate cell while stage 3 or 4 is running** (Colab lets you
queue a cell; it executes after the current one finishes) - or better, from the
Colab terminal, where it runs immediately alongside the build.

What to expect:

| stage | cores used | why |
|---|---|---|
| 1 | 1 | trivial, seconds |
| **2** | **1** | serial by design - ~13 min pinned to one core, this is normal |
| 3 | all | one kit subprocess per worker; bounded by ~98 chunks, so the last wave tails off |
| 4 | all | ~129,000 independent jobs, no bound |


In [ ]:
# Snapshot of what is actually running. Safe to run at any time.
import os, subprocess, time

ncpu = os.cpu_count()
la1, la5, la15 = os.getloadavg()
print(f'vCPUs visible            {ncpu}')
print(f'ECG_WORKERS              {os.environ.get("ECG_WORKERS")}')
print(f'load average (1/5/15m)   {la1:.1f} / {la5:.1f} / {la15:.1f}')
print(f'  -> {100*la1/ncpu:.0f}% of {ncpu} cores busy (1 min avg)')

# count live python workers, excluding this notebook kernel
ps = subprocess.run(['ps','-eo','pid,ppid,pcpu,etimes,comm','--no-headers'],
                    capture_output=True, text=True).stdout.splitlines()
busy = [l.split(None,4) for l in ps if float(l.split()[2]) > 20.0]
print(f'\nprocesses above 20% CPU: {len(busy)}')
for pid, ppid, pcpu, et, comm in busy[:20]:
    print(f'  pid {pid:>7s}  {float(pcpu):5.1f}%  {int(et):5d}s  {comm}')

# progress by artefact count - more trustworthy than any log line
for name, pat, target in (('rendered PNG', 'rendered/**/*.png', 37225),
                          ('final JPEG',   'images/**/*.jpg',   128994)):
    n = len(list((BUILD).glob(pat)))
    print(f'{name:14s} {n:>7,} / {target:,}  ({100*n/target:5.1f}%)')


## 4. Package to Drive

**Never write 129k loose files to Drive.** Each one is a FUSE round-trip and you
will hit the API rate limit long before it finishes. Tar into a few large
archives instead: a handful of calls rather than a quarter of a million.


In [ ]:
idx = pd.read_csv(BUILD/'index.csv')
print(f'{len(idx):,} images')
tot = sum(f.stat().st_size for f in (BUILD/'images').rglob('*') if f.is_file())
print(f'{tot/2**30:.1f} GB to upload')
free_drive = None  # check your Google One quota; 117.8 GB needs the 200 GB plan

EXPDIR.mkdir(parents=True, exist_ok=True); RUNDIR.mkdir(parents=True, exist_ok=True)
# small artifacts first, so a later failure still leaves the manifest on Drive
idx.to_csv(RUNDIR/'index.csv', index=False)
(RUNDIR/'settings.json').write_text(json.dumps({
    'bandpass_hz': list(C.BANDPASS), 'dpi': C.DPI, 'print_header': C.PRINT_HEADER,
    'annotations': C.ANNOTATIONS, 'row_gap': C.ROW_GAP, 'rhythm_gap': C.RHYTHM_GAP,
    'variants': C.VARIANTS_PER_SPLIT, 'records': int(len(man)), 'images': int(len(idx)),
    'hours': round((time.time()-t0)/3600, 2)}, indent=2))
for name, txt in (('stage1.log',out1),('stage2.log',out2),('stage5.log',out5)):
    (RUNDIR/name).write_text(txt)
print('wrote index + settings + logs to', RUNDIR)


In [ ]:
# One archive per split. test/val are small; train is the bulk.
for split in ('test','val','train'):
    src = BUILD/'images'/split
    if not src.is_dir():
        continue
    dst = EXPDIR/f'images_{split}.tar'
    if dst.exists():
        print('skip (exists):', dst.name)
        continue
    print(f'packing {split} ...', flush=True)
    r = subprocess.run(['tar','-cf',str(dst),'-C',str(src),'.'],
                       capture_output=True, text=True)
    if r.returncode:
        print(r.stderr[-1000:]); raise SystemExit(f'tar failed for {split}')
    print(f'  {dst.name}  {dst.stat().st_size/2**30:.1f} GB')
for t in sorted(EXPDIR.glob('images_*.tar')):
    print(f'  {t.name:24s} {t.stat().st_size/2**30:7.1f} GB')


In [ ]:
# Checksums, so a truncated upload is detectable later.
import hashlib
for t in sorted(EXPDIR.glob('images_*.tar')):
    s = EXPDIR/(t.name + '.sha256')
    if s.exists():
        continue
    h = hashlib.sha256()
    with open(t,'rb') as fh:
        for chunk in iter(lambda: fh.read(1<<24), b''):
            h.update(chunk)
    s.write_text(f'{h.hexdigest()}  {t.name}\n')
    print(s.name, 'written')
print('\nDONE.')
print('  corpus archives ->', EXPDIR)
print('  run metadata    ->', RUNDIR)
